#1. Imports

In [1]:
import os, pickle, datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE

#2. Configuration

In [2]:
MODEL_DIR    = "models"
LOG_DIR      = "logs"
DATA_DIR     = "data"

FEATURES = [
    "latitude",
    "longitude",
    "competitor_density_500m",
    "jarak_kompetitor_meter",
    "kompetitor_head_to_head",
    "jarak_pasar_meter",
    "cluster_hdbscan_makro",
    "cluster_hdbscan_prob",
    "is_hotspot",
    "density_x_headtohead",
    "density_per_distance",
]
TARGET       = "pelanggaran_zonasi"

# GBM config
GBM_N_ESTIMATORS  = 300
GBM_LEARNING_RATE = 0.05
GBM_MAX_DEPTH     = 4

# TF config
EPOCHS           = 200
BATCH_SIZE       = 32
LEARNING_RATE    = 2e-4
VIOLATION_WEIGHT = 1.0
PATIENCE         = 40

# Shared
TEST_SIZE    = 0.2
RANDOM_STATE = 42
THRESHOLD    = 0.3

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print("✅ Config ready")

✅ Config ready


#3. Load & Validate Data

In [3]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/shineistu86/CPS-CC26/c57a4ed93bd73d13f92b8aff1c396f16b57daa7b/Data%20Clean/jaksel_spatial_features_v6_AI_ready.csv"
)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Violations: {df[TARGET].sum()} / {len(df)}")
print(f"Missing values:\n{df[FEATURES[:9]].isnull().sum()}")

Shape: (662, 17)
Columns: ['store', 'nama_tempat', 'latitude', 'longitude', 'competitor_density_500m', 'jarak_kompetitor_meter', 'kompetitor_head_to_head', 'cluster_hdbscan_prob', 'is_hotspot', 'pasar_density_500m', 'retail_vs_pasar_ratio', 'pelanggaran_zonasi', 'jarak_pasar_meter', 'pasar_terdekat', 'spatial_risk_score', 'cluster_dbscan_raw', 'cluster_hdbscan_makro']
Violations: 99 / 662
Missing values:
latitude                   0
longitude                  0
competitor_density_500m    0
jarak_kompetitor_meter     0
kompetitor_head_to_head    0
jarak_pasar_meter          0
cluster_hdbscan_makro      0
cluster_hdbscan_prob       0
is_hotspot                 0
dtype: int64


#4. Feature Engineering

In [4]:
df[TARGET] = df[TARGET].astype(int)

RAW_FEATURES = [
    "latitude", "longitude", "competitor_density_500m",
    "jarak_kompetitor_meter", "kompetitor_head_to_head",
    "jarak_pasar_meter", "cluster_hdbscan_makro",
    "cluster_hdbscan_prob", "is_hotspot",
]
df = df.dropna(subset=RAW_FEATURES + [TARGET])

# Engineered features
df["density_x_headtohead"] = (
    df["competitor_density_500m"] * df["kompetitor_head_to_head"]
)
df["density_per_distance"] = (
    df["competitor_density_500m"] / (df["jarak_kompetitor_meter"] + 1)
)

print(f"Rows after dropna: {len(df)}")
print(f"Class distribution: {df[TARGET].value_counts().to_dict()}")

Rows after dropna: 662
Class distribution: {0: 563, 1: 99}


#5. Correlation Check

In [5]:
corrs = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET)
corrs = corrs.abs().sort_values(ascending=False)

print("Feature correlations with target (absolute):")
print("=" * 50)
for feat, corr in corrs.items():
    bar = "█" * int(corr * 40)
    print(f"{feat:<30} {corr:.4f}  {bar}")

Feature correlations with target (absolute):
jarak_pasar_meter              0.5698  ██████████████████████
density_per_distance           0.1009  ████
latitude                       0.0988  ███
cluster_hdbscan_makro          0.0583  ██
kompetitor_head_to_head        0.0464  █
cluster_hdbscan_prob           0.0440  █
density_x_headtohead           0.0328  █
is_hotspot                     0.0213  
longitude                      0.0187  
jarak_kompetitor_meter         0.0054  
competitor_density_500m        0.0014  


#6. Extract X, y

In [6]:
X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Violation rate: {y.mean():.3f}")

X shape: (662, 11)
y shape: (662,)
Violation rate: 0.150


#7. Stratified Train/Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train distribution: {pd.Series(y_train).value_counts().to_dict()}")
print(f"y_test  distribution: {pd.Series(y_test).value_counts().to_dict()}")

X_train: (529, 11) | X_test: (133, 11)
y_train distribution: {0.0: 450, 1.0: 79}
y_test  distribution: {0.0: 113, 1.0: 20}


#8. Save Split Data

In [8]:
pd.DataFrame(X_train, columns=FEATURES).assign(
    pelanggaran_zonasi=y_train
).to_csv(f"{DATA_DIR}/X_train.csv", index=False)

pd.DataFrame(X_test, columns=FEATURES).assign(
    pelanggaran_zonasi=y_test
).to_csv(f"{DATA_DIR}/X_test.csv", index=False)

print(f"✅ Saved X_train.csv ({len(X_train)} rows)")
print(f"✅ Saved X_test.csv  ({len(X_test)} rows)")

✅ Saved X_train.csv (529 rows)
✅ Saved X_test.csv  (133 rows)


#9. SMOTE — Oversampling (BEFORE scaling)

In [9]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {pd.Series(y_train).value_counts().to_dict()}")
print(f"After SMOTE : {pd.Series(y_train_sm).value_counts().to_dict()}")
print(f"X_train_sm shape: {X_train_sm.shape}")

Before SMOTE: {0.0: 450, 1.0: 79}
After SMOTE : {1.0: 450, 0.0: 450}
X_train_sm shape: (900, 11)


#10. Feature Scaling (AFTER SMOTE)

In [10]:
# Fit scaler on SMOTE-augmented training data, transform test separately

scaler     = StandardScaler()
X_train_sm = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test)          # never fit on test!

with open(f"{MODEL_DIR}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Sanity check — means should be ~0 for both
print(f"X_train_sm mean (should be ~0): {X_train_sm.mean(axis=0).round(3)}")
print(f"X_test_sc  mean (should be ~0): {X_test_sc.mean(axis=0).round(3)}")
print(f"✅ Scaler saved to {MODEL_DIR}/scaler.pkl")

# Cast
X_train_sm = X_train_sm.astype(np.float32)
X_test_sc  = X_test_sc.astype(np.float32)
y_train_sm = y_train_sm.astype(np.float32)
y_test     = y_test.astype(np.float32)

X_train_sm mean (should be ~0): [ 0.  0. -0.  0.  0.  0. -0. -0.  0.  0. -0.]
X_test_sc  mean (should be ~0): [-0.207  0.06   0.058 -0.012  0.161  0.514 -0.327 -0.277  0.158  0.265
 -0.14 ]
✅ Scaler saved to models/scaler.pkl


# 11. Gradient Boosting Classifier

In [11]:
gb_model = GradientBoostingClassifier(
    n_estimators        = GBM_N_ESTIMATORS,
    learning_rate       = GBM_LEARNING_RATE,
    max_depth           = GBM_MAX_DEPTH,
    min_samples_leaf    = 10,
    subsample           = 0.8,
    max_features        = "sqrt",
    random_state        = RANDOM_STATE,
    validation_fraction = 0.1,
    n_iter_no_change    = 20,
    tol                 = 1e-4,
)

gb_model.fit(X_train_sm, y_train_sm)
print(f"✅ GBM trained | estimators used: {gb_model.n_estimators_}")

✅ GBM trained | estimators used: 250


# 12. GBM Evaluation

In [12]:
y_prob_gb = gb_model.predict_proba(X_test_sc)[:, 1]
y_pred_gb = (y_prob_gb >= THRESHOLD).astype(int)

acc_gb         = (y_pred_gb == y_test.astype(int)).mean()
mae_rounded_gb = np.abs(y_pred_gb - y_test.astype(int)).mean()
mae_raw_gb     = np.abs(y_prob_gb - y_test).mean()
auc_gb         = roc_auc_score(y_test, y_prob_gb)
f1_gb          = f1_score(y_test, y_pred_gb)

print("=" * 50)
print("GRADIENT BOOSTING RESULTS")
print("=" * 50)
print(f"Accuracy      : {acc_gb:.4f}")
print(f"MAE (rounded) : {mae_rounded_gb:.4f}  ← target <= 0.02")
print(f"MAE (raw prob): {mae_raw_gb:.4f}")
print(f"AUC-ROC       : {auc_gb:.4f}")
print(f"F1 Score      : {f1_gb:.4f}")
print("=" * 50)
print(classification_report(y_test, y_pred_gb, target_names=["Compliant", "Violation"]))

# Feature importance
importance_df = pd.DataFrame({
    "feature"   : FEATURES,
    "importance": gb_model.feature_importances_,
}).sort_values("importance", ascending=False)

print("Feature Importances:")
print(importance_df.to_string(index=False))

with open(f"{MODEL_DIR}/gb_model.pkl", "wb") as f:
    pickle.dump(gb_model, f)
print(f"\n✅ Saved: {MODEL_DIR}/gb_model.pkl")

GRADIENT BOOSTING RESULTS
Accuracy      : 1.0000
MAE (rounded) : 0.0000  ← target <= 0.02
MAE (raw prob): 0.0001
AUC-ROC       : 1.0000
F1 Score      : 1.0000
              precision    recall  f1-score   support

   Compliant       1.00      1.00      1.00       113
   Violation       1.00      1.00      1.00        20

    accuracy                           1.00       133
   macro avg       1.00      1.00      1.00       133
weighted avg       1.00      1.00      1.00       133

Feature Importances:
                feature  importance
      jarak_pasar_meter    0.799420
  cluster_hdbscan_makro    0.037405
   cluster_hdbscan_prob    0.034415
               latitude    0.033847
              longitude    0.027342
competitor_density_500m    0.022851
             is_hotspot    0.020000
 jarak_kompetitor_meter    0.017802
   density_per_distance    0.006639
kompetitor_head_to_head    0.000203
   density_x_headtohead    0.000076

✅ Saved: models/gb_model.pkl


# 13. TensorFlow — Custom Layer, Loss & Metric

In [13]:
class SpatialDensityEmbedding(tf.keras.layers.Layer):
    def __init__(self, units=32, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.dense = tf.keras.layers.Dense(units, activation="relu")

    def build(self, input_shape):
        n_features = input_shape[-1]
        self.feature_attention = tf.keras.layers.Dense(
            n_features, activation="sigmoid"
        )
        self.feature_attention.build(input_shape)
        super().build(input_shape)

    def call(self, inputs, training=False):
        attention = self.feature_attention(inputs)
        weighted  = inputs * attention
        return self.dense(weighted)

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config


def zonasi_custom_loss(y_true, y_pred):
    epsilon = 1e-7
    y_pred  = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
    bce = -(y_true * tf.math.log(y_pred) +
            (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    weights = tf.where(y_true == 1.0, VIOLATION_WEIGHT, 1.0)
    return tf.reduce_mean(bce * weights)


class RoundedMAE(tf.keras.metrics.Metric):
    def __init__(self, threshold=THRESHOLD, name="rounded_mae", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.total = self.add_weight(name="total", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_rounded = tf.cast(y_pred >= self.threshold, tf.float32)
        mae = tf.abs(y_true - y_pred_rounded)
        self.total.assign_add(tf.reduce_sum(mae))
        self.count.assign_add(tf.cast(tf.size(y_true), tf.float32))

    def result(self):
        return self.total / self.count

    def reset_state(self):
        self.total.assign(0.0)
        self.count.assign(0.0)

print("✅ Custom classes defined")

✅ Custom classes defined


# 14. TensorFlow — Model Architecture

In [14]:
def build_model(input_dim: int) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(input_dim,), name="spatial_features")
    x = SpatialDensityEmbedding(units=64, name="spatial_embedding")(inputs)
    x = tf.keras.layers.BatchNormalization(name="bn_1")(x)
    x = tf.keras.layers.Dense(128, activation="relu", name="dense_1")(x)
    x = tf.keras.layers.Dropout(0.3, name="dropout_1")(x)
    x = tf.keras.layers.Dense(64, activation="relu", name="dense_2")(x)
    x = tf.keras.layers.BatchNormalization(name="bn_2")(x)
    x = tf.keras.layers.Dropout(0.2, name="dropout_2")(x)
    x = tf.keras.layers.Dense(32, activation="relu", name="dense_3")(x)
    output = tf.keras.layers.Dense(1, activation="sigmoid", name="violation_prob")(x)
    return tf.keras.Model(inputs=inputs, outputs=output, name="ZonifyModel_v2")

model = build_model(input_dim=len(FEATURES))
model.summary()

Model: "ZonifyModel_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ spatial_features (InputLayer)   │ (None, 11)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_embedding               │ (None, 64)             │           900 │
│ (SpatialDensityEmbedding)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ violation_prob (Dense)          │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,101 (78.52 KB)

 Trainable params: 19,845 (77.52 KB)

 Non-trainable params: 256 (1.00 KB)

# 15. TensorFlow — Optimizer, Metrics & Dataset Pipeline

In [15]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

train_loss_metric = tf.keras.metrics.Mean(name="train_loss")
train_acc_metric  = tf.keras.metrics.BinaryAccuracy(name="train_acc", threshold=THRESHOLD)
train_mae_metric  = RoundedMAE(name="train_mae")
val_loss_metric   = tf.keras.metrics.Mean(name="val_loss")
val_acc_metric    = tf.keras.metrics.BinaryAccuracy(name="val_acc", threshold=THRESHOLD)
val_mae_metric    = RoundedMAE(name="val_mae")

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_train_sm, y_train_sm))
    .shuffle(buffer_size=len(X_train_sm), seed=RANDOM_STATE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_test_sc, y_test))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

run_name  = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path  = os.path.join(LOG_DIR, run_name)
tb_writer = tf.summary.create_file_writer(log_path)

print(f"Train batches : {len(train_dataset)}")
print(f"Val   batches : {len(val_dataset)}")

Train batches : 29
Val   batches : 5


# 16. TensorFlow — Train & Val Step Functions

In [16]:
@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predictions = model(x_batch, training=True)
        predictions = tf.squeeze(predictions, axis=-1)
        loss = zonasi_custom_loss(y_batch, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    train_loss_metric(loss)
    train_acc_metric(y_batch, predictions)
    train_mae_metric(y_batch, predictions)

@tf.function
def val_step(x_batch, y_batch):
    predictions = model(x_batch, training=False)
    predictions = tf.squeeze(predictions, axis=-1)
    loss = zonasi_custom_loss(y_batch, predictions)
    val_loss_metric(loss)
    val_acc_metric(y_batch, predictions)
    val_mae_metric(y_batch, predictions)

print("✅ Step functions ready")

✅ Step functions ready


# 17. TensorFlow — Training Loop

In [17]:
best_val_mae     = float("inf")
best_val_acc     = 0.0
patience_counter = 0

print(f"{'Epoch':>6} | {'TrainLoss':>9} | {'TrainAcc':>8} | {'TrainMAE':>8} | {'ValLoss':>8} | {'ValAcc':>7} | {'ValMAE':>7}")
print("-" * 75)

for epoch in range(1, EPOCHS + 1):
    train_loss_metric.reset_state()
    train_acc_metric.reset_state()
    train_mae_metric.reset_state()
    val_loss_metric.reset_state()
    val_acc_metric.reset_state()
    val_mae_metric.reset_state()

    for x_batch, y_batch in train_dataset:
        train_step(x_batch, y_batch)

    for x_batch, y_batch in val_dataset:
        val_step(x_batch, y_batch)

    t_loss = train_loss_metric.result().numpy()
    t_acc  = train_acc_metric.result().numpy()
    t_mae  = train_mae_metric.result().numpy()
    v_loss = val_loss_metric.result().numpy()
    v_acc  = val_acc_metric.result().numpy()
    v_mae  = val_mae_metric.result().numpy()

    with tb_writer.as_default():
        tf.summary.scalar("accuracy/train", t_acc,  step=epoch)
        tf.summary.scalar("accuracy/val",   v_acc,  step=epoch)
        tf.summary.scalar("loss/train",     t_loss, step=epoch)
        tf.summary.scalar("loss/val",       v_loss, step=epoch)
        tf.summary.scalar("mae/train",      t_mae,  step=epoch)
        tf.summary.scalar("mae/val",        v_mae,  step=epoch)

    print(f"{epoch:>6} | {t_loss:>9.4f} | {t_acc:>8.4f} | {t_mae:>8.4f} | {v_loss:>8.4f} | {v_acc:>7.4f} | {v_mae:>7.4f}")

    # Save best by MAE (primary), then acc (secondary)
    if v_mae < best_val_mae or (v_mae == best_val_mae and v_acc > best_val_acc):
        best_val_mae = v_mae
        best_val_acc = v_acc
        model.save(f"{MODEL_DIR}/zonify_model.keras")
        patience_counter = 0
        print(f"         ✓ Saved (val_mae={v_mae:.4f}, val_acc={v_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n[EARLY STOP] {PATIENCE} epochs no improvement.")
            break

 Epoch | TrainLoss | TrainAcc | TrainMAE |  ValLoss |  ValAcc |  ValMAE
---------------------------------------------------------------------------
     1 |    0.7825 |   0.5356 |   0.4644 |   0.6657 |  0.1504 |  0.8496
         ✓ Saved (val_mae=0.8496, val_acc=0.1504)
     2 |    0.6404 |   0.6100 |   0.3900 |   0.6594 |  0.1504 |  0.8496
     3 |    0.5304 |   0.6889 |   0.3111 |   0.6524 |  0.1504 |  0.8496
     4 |    0.4607 |   0.7322 |   0.2678 |   0.6334 |  0.1504 |  0.8496
     5 |    0.4029 |   0.7956 |   0.2044 |   0.6118 |  0.1504 |  0.8496
     6 |    0.4098 |   0.7956 |   0.2044 |   0.5805 |  0.1729 |  0.8271
         ✓ Saved (val_mae=0.8271, val_acc=0.1729)
     7 |    0.3455 |   0.8233 |   0.1767 |   0.5333 |  0.2707 |  0.7293
         ✓ Saved (val_mae=0.7293, val_acc=0.2707)
     8 |    0.3019 |   0.8444 |   0.1556 |   0.4884 |  0.3835 |  0.6165
         ✓ Saved (val_mae=0.6165, val_acc=0.3835)
     9 |    0.2875 |   0.8578 |   0.1422 |   0.4372 |  0.5564 |  0.4436
    

# 18. TensorFlow — Final Evaluation

In [18]:
best_model = tf.keras.models.load_model(
    f"{MODEL_DIR}/zonify_model.keras",
    custom_objects={
        "SpatialDensityEmbedding": SpatialDensityEmbedding,
        "zonasi_custom_loss"      : zonasi_custom_loss,
        "RoundedMAE"              : RoundedMAE,
    },
)

y_prob_tf = best_model.predict(X_test_sc, verbose=0).flatten()
y_pred_tf = (y_prob_tf >= THRESHOLD).astype(int)

acc_tf         = (y_pred_tf == y_test.astype(int)).mean()
mae_rounded_tf = np.abs(y_pred_tf - y_test.astype(int)).mean()
mae_raw_tf     = np.abs(y_prob_tf - y_test).mean()
auc_tf         = roc_auc_score(y_test, y_prob_tf)
f1_tf          = f1_score(y_test, y_pred_tf)

print("=" * 50)
print("TENSORFLOW RESULTS")
print("=" * 50)
print(f"Accuracy      : {acc_tf:.4f}")
print(f"MAE (rounded) : {mae_rounded_tf:.4f}  ← target <= 0.02")
print(f"MAE (raw prob): {mae_raw_tf:.4f}")
print(f"AUC-ROC       : {auc_tf:.4f}")
print(f"F1 Score      : {f1_tf:.4f}")
print("=" * 50)
print(classification_report(y_test, y_pred_tf, target_names=["Compliant", "Violation"]))

TENSORFLOW RESULTS
Accuracy      : 1.0000
MAE (rounded) : 0.0000  ← target <= 0.02
MAE (raw prob): 0.0140
AUC-ROC       : 1.0000
F1 Score      : 1.0000
              precision    recall  f1-score   support

   Compliant       1.00      1.00      1.00       113
   Violation       1.00      1.00      1.00        20

    accuracy                           1.00       133
   macro avg       1.00      1.00      1.00       133
weighted avg       1.00      1.00      1.00       133



# 19. Model Comparison Summary

In [19]:
print("=" * 65)
print("  MODEL COMPARISON")
print("=" * 65)
print(f"{'Metric':<20} {'GradientBoosting':>20} {'TensorFlow':>20}")
print("-" * 65)
metrics = [
    ("Accuracy",       f"{acc_gb:.4f}",         f"{acc_tf:.4f}"),
    ("MAE (rounded)",  f"{mae_rounded_gb:.4f}",  f"{mae_rounded_tf:.4f}"),
    ("MAE (raw prob)", f"{mae_raw_gb:.4f}",      f"{mae_raw_tf:.4f}"),
    ("AUC-ROC",        f"{auc_gb:.4f}",          f"{auc_tf:.4f}"),
    ("F1 Score",       f"{f1_gb:.4f}",           f"{f1_tf:.4f}"),
]
for label, gbm_val, tf_val in metrics:
    print(f"{label:<20} {gbm_val:>20} {tf_val:>20}")
print("=" * 65)

  MODEL COMPARISON
Metric                   GradientBoosting           TensorFlow
-----------------------------------------------------------------
Accuracy                           1.0000               1.0000
MAE (rounded)                      0.0000               0.0000
MAE (raw prob)                     0.0001               0.0140
AUC-ROC                            1.0000               1.0000
F1 Score                           1.0000               1.0000


# 20. Model Load Test & API Simulation

In [20]:
with open(f"{MODEL_DIR}/scaler.pkl", "rb") as f:
    loaded_scaler = pickle.load(f)
with open(f"{MODEL_DIR}/gb_model.pkl", "rb") as f:
    loaded_gb = pickle.load(f)
loaded_tf = tf.keras.models.load_model(
    f"{MODEL_DIR}/zonify_model.keras",
    custom_objects={
        "SpatialDensityEmbedding": SpatialDensityEmbedding,
        "zonasi_custom_loss"      : zonasi_custom_loss,
        "RoundedMAE"              : RoundedMAE,
    },
)
print("✅ All models and scaler loaded")

# Sample input
lat                     = -6.2363
lng                     = 106.8568
competitor_density_500m = 5
jarak_kompetitor_meter  = 162.63
kompetitor_head_to_head = 0
jarak_pasar_meter       = 350.0
cluster_hdbscan_makro   = 2
cluster_hdbscan_prob    = 0.85
is_hotspot              = 1
density_x_headtohead    = competitor_density_500m * kompetitor_head_to_head
density_per_distance    = competitor_density_500m / (jarak_kompetitor_meter + 1)

sample = np.array([[
    lat, lng, competitor_density_500m, jarak_kompetitor_meter,
    kompetitor_head_to_head, jarak_pasar_meter, cluster_hdbscan_makro,
    cluster_hdbscan_prob, is_hotspot, density_x_headtohead, density_per_distance
]], dtype=np.float32)

scaled = loaded_scaler.transform(sample)

prob_gb = loaded_gb.predict_proba(scaled)[0][1]
pred_gb = int(prob_gb >= THRESHOLD)
prob_tf = float(loaded_tf.predict(scaled, verbose=0)[0][0])
pred_tf = int(prob_tf >= THRESHOLD)

print(f"\nSample input shape: {sample.shape}")
print(f"\n{'Model':<20} {'Probability':>12} {'Verdict':>15}")
print("-" * 50)
print(f"{'GradientBoosting':<20} {prob_gb:>12.4f} {'⚠️ PELANGGARAN' if pred_gb else '✅ PATUH':>15}")
print(f"{'TensorFlow':<20} {prob_tf:>12.4f} {'⚠️ PELANGGARAN' if pred_tf else '✅ PATUH':>15}")

✅ All models and scaler loaded

Sample input shape: (1, 11)

Model                 Probability         Verdict
--------------------------------------------------
GradientBoosting           0.9999  ⚠️ PELANGGARAN
TensorFlow                 0.9965  ⚠️ PELANGGARAN
